In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc
)

from torchvision import datasets
from torch.utils.data import DataLoader

import timm
import torch.nn as nn
from timm.data import resolve_data_config, create_transform


# =========================================================
# CONFIGURATION
# =========================================================
class CFG:
    test_dir = "./data/Test"
    model_name = "swin_small_patch4_window7_224"
    batch_size = 32

    device = "cuda" if torch.cuda.is_available() else "cpu"

    checkpoint_dir = "./checkpoints"
    model_file = "lung_swin_classifier_best.pth"

    output_dir = "./outputs"


cfg = CFG()
os.makedirs(cfg.output_dir, exist_ok=True)


# =========================================================
# MODEL (EXACT SAME AS TRAINING)
# =========================================================
class Head(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 2)
        )

    def forward(self, x):
        return self.net(x)


base_model = timm.create_model(cfg.model_name, pretrained=False, num_classes=0)
model = nn.Sequential(base_model, Head(base_model.num_features)).to(cfg.device)


# =========================================================
# LOAD MODEL
# =========================================================
checkpoint = torch.load(
    os.path.join(cfg.checkpoint_dir, cfg.model_file),
    map_location=cfg.device
)

model.load_state_dict(checkpoint["model"])
model.eval()

print("Model loaded successfully")


# =========================================================
# DATA
# =========================================================
data_cfg = resolve_data_config({}, model=base_model)
transform = create_transform(**data_cfg, is_training=False)

test_data = datasets.ImageFolder(cfg.test_dir, transform=transform)
test_loader = DataLoader(test_data, batch_size=cfg.batch_size, shuffle=False)

class_names = test_data.classes

print(f"Test samples: {len(test_data)}")


# =========================================================
# PREDICTION
# =========================================================
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(cfg.device)
        y = y.to(cfg.device)

        out = model(x)
        prob = torch.softmax(out, dim=1)

        preds = torch.argmax(out, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        all_probs.extend(prob[:, 1].cpu().numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)
y_prob = np.array(all_probs)

print("Evaluation completed")


# =========================================================
# CONFUSION MATRIX
# =========================================================
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Reds",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, "confusion_matrix.png"))
plt.show()


# =========================================================
# ROC CURVE
# =========================================================
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, "roc_curve.png"))
plt.show()


# =========================================================
# METRICS
# =========================================================
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="macro")
recall = recall_score(y_true, y_pred, average="macro")
f1 = f1_score(y_true, y_pred, average="macro")

tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("\n===== EVALUATION METRICS =====")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"Sensitivity  : {sensitivity:.4f}")
print(f"Specificity  : {specificity:.4f}")
print(f"AUC          : {roc_auc:.4f}")


# =========================================================
# CLASSIFICATION REPORT
# =========================================================
print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_true, y_pred, target_names=class_names))